# Nonlocal Low-Rank Denoiser (2D) — **GPU** (PyTorch CUDA)

**Pipeline**
1) For each anchor (stride = `anchor_step`), collect candidates in a `(2r+1)²` window.  
2) k-NN (mean-normalized L2) to pick `K` similar patches.  
3) Stack as `P×K` (`P = patch²`), center (per-patch / per-group / none), run **SVD on GPU**.  
4) **Shrink per component** (choose: Soft / WNNM / Wiener; all **σ-aware**), invert, overlap-add with **Kaiser** window.

**Why GPU here?** The per-group SVD and k-NN distances are the heavy ops; on CUDA they’re fast and VRAM-light (typical `64×24`).

**Notes**
- Input must be 2D grayscale `float32` in `[0,1]` (convert beforehand).  
- Designed for 16 GB VRAM; only small group matrices go to GPU.


In [1]:
from __future__ import annotations

import os, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_laplace

# (Optional) skimage bits for convenience — not required for the core
try:
    from skimage.restoration import estimate_sigma as sk_estimate_sigma
    from skimage import color, img_as_float32, io, exposure
    from skimage.util import view_as_windows
except Exception:
    sk_estimate_sigma = None
    color = img_as_float32 = io = exposure = view_as_windows = None

torch.set_grad_enabled(False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

DATA_DIR = "/home/askiran/data/"

Device: cuda


## Configuration (edit here)

**Parameters**
- `patch` *(int)*: patch width `a` (6–10 typical; default 8).
- `search_radius` *(int)*: half-window `r` for k-NN (12–24 typical; default 16).
- `K` *(int)*: # of neighbors (16–32 typical; default 24).  
- `anchor_step` *(int)*: anchor stride (↑ faster; default 4).
- `method` *(str)*: `'soft'` (NNM), `'wnnm'` (Weighted NNM), `'wiener'` (PCA-Wiener).
- `tau` *(float|None)*: soft-threshold; if `None`, auto from `sigma`.
- `wnnm_c` *(float)*: WNNM scale (0.6–1.5 typical; default 1.0).
- `sigma` *(float|None)*: noise std in `[0,1]`; if `None`, robustly estimated.
- `mean_normalize` *(bool)*: mean-normalize patches for distance (robust for textures).
- `center_mode` *('patch'|'group'|'none')*: centering before SVD.
- `kaiser_beta` *(float)*: overlap window softness (1.5–3.0 typical; default 2.0).
- `trim_frac` *(float)*: optionally drop worst fraction of K neighbors **after** sorting (e.g., 0.1 keeps best 90%).

**Auto-τ**: `tau = sigma * sqrt(2 * max(P, K))`, `P = patch²`.

**Tuning**
- Detail loss → ↓τ or `method='wnnm'` with `wnnm_c≈0.7–0.9`.  
- Residual noise → ↑τ ~10–20% or ↑K (and maybe ↑r).  
- Speed → ↑`anchor_step` (8 during tuning), ↓`r`, ↓`K`.  
- Robustness → set `trim_frac∈[0,0.2]` to reject distant neighbors.


In [2]:
cfg = {
    "patch": 8,
    "search_radius": 16,
    "K": 24,
    "anchor_step": 4,
    "method": "soft",          # 'soft' | 'wnnm' | 'wiener'
    "tau": None,               # None => auto
    "wnnm_c": 1.0,
    "sigma": None,             # None => estimate
    "mean_normalize": True,
    "center_mode": "patch",    # 'patch' | 'group' | 'none'
    "kaiser_beta": 2.0,
    "trim_frac": 0.0,          # e.g., 0.1 drops worst 10% neighbors
    "dtype": np.float32,
}
print("cfg set ✅")


cfg set ✅


- Set `img_path` to your file.
- Handles TIFF/OME (optionally pick `page`), PNG/JPG, etc.
- Converts to **grayscale float32 in [0,1]** and asserts it’s 2D.

In [4]:
import os, numpy as np
from skimage import io, img_as_float32, color

# ==== Set your image path here ====
img_path = "/home/askiran/data/Peri_1/stich_pixel_0pt079.tif"   # <-- change this
page = None  # set an int (e.g., 0) if you want a specific TIFF/OME page

def load_img_to_img0(path: str, page=None) -> np.ndarray:
    p = str(path)
    ext = os.path.splitext(p)[1].lower()
    # Read
    if ext in (".tif", ".tiff") or ".ome.tif" in p.lower() or ".ome.tiff" in p.lower():
        try:
            import tifffile as tiff
            arr = tiff.imread(p) if page is None else tiff.imread(p, key=int(page))
        except Exception:
            arr = io.imread(p)
    else:
        arr = io.imread(p)

    # Select slice for (Z,H,W) stacks if needed
    if arr.ndim == 3 and arr.shape[-1] not in (3,4) and arr.shape[0] >= 2:
        z = 0 if page is None else int(page)
        arr = arr[z]

    # To grayscale if RGB/RGBA
    if arr.ndim == 3 and arr.shape[-1] in (3,4):
        try:
            arr = color.rgb2gray(arr[..., :3])
        except Exception:
            arr = arr[..., :3].mean(axis=-1)

    # Float32 in [0,1], contiguous
    img = img_as_float32(arr)
    img = np.clip(img, 0.0, 1.0)
    assert img.ndim == 2, f"Expected 2D image, got shape {img.shape}"
    return np.ascontiguousarray(img, dtype=np.float32)

img0 = load_img_to_img0(img_path, page)
print("img0 ready:", img0.shape, img0.dtype, float(img0.min()), float(img0.max()))


img0 ready: (3124, 3232) float32 0.0 1.0


## Utilities

- `estimate_sigma(img)`: noise estimator — `skimage` if present; fallback MAD on LoG.
- `kaiser2d(a, beta)`: 2D Kaiser for smooth overlap.
- `_to_gray(img)`: convert H×W×C to luminance if needed.

**Caveat**: If your noise is not i.i.d. Gaussian, set `cfg["sigma"]` manually.


In [5]:
def estimate_sigma(img: np.ndarray, fallback_log_sigma: float = 1.2) -> float:
    if sk_estimate_sigma is not None:
        try:
            s = float(sk_estimate_sigma(img, average_sigmas=True, channel_axis=None))
            if np.isfinite(s) and s > 0:
                return s
        except Exception:
            pass
    hp = gaussian_laplace(img, sigma=fallback_log_sigma)
    mad = np.median(np.abs(hp - np.median(hp)))
    return float(1.4826 * mad)

def kaiser2d(a: int, beta: float = 2.0, dtype=np.float32) -> np.ndarray:
    w1d = np.kaiser(a, beta).astype(dtype)
    w2d = np.outer(w1d, w1d)
    w2d /= (w2d.max() + 1e-12)
    return w2d

def _to_gray(img: np.ndarray) -> np.ndarray:
    if img.ndim == 2: return img
    if color is not None and img.ndim == 3 and img.shape[-1] in (3,4):
        return color.rgb2gray(img[..., :3])
    if img.ndim == 3: return img.mean(axis=-1)
    raise ValueError(f"Expected 2D or HxWx3/4; got {img.shape}")


## GPU k-NN distances & centering helpers

- `extract_window_patches(img, y, x, a, r)`: returns `(view, top_lefts)` with `view.shape=(Ny, Nx, a, a)`.
- `topk_similar_torch(view, anchor, K, mean_normalize)`: sends candidates/anchor to CUDA; returns **sorted** top-K indices (smallest L2).
- `center_for_svd(M, mode)`: how to center group matrix `P×K` before SVD.


In [6]:
def extract_window_patches(img: np.ndarray, y: int, x: int, a: int, r: int):
    H, W = img.shape
    y0 = max(0, y - r); y1 = min(H - a, y + r)
    x0 = max(0, x - r); x1 = min(W - a, x + r)
    region = img[y0:y1 + a, x0:x1 + a]
    if view_as_windows is not None:
        view = view_as_windows(region, (a, a))  # (Ny, Nx, a, a)
    else:
        Ny = (region.shape[0] - a) + 1
        Nx = (region.shape[1] - a) + 1
        view = np.zeros((Ny, Nx, a, a), dtype=img.dtype)
        for iy in range(Ny):
            for ix in range(Nx):
                view[iy, ix] = region[iy:iy+a, ix:ix+a]
    Ny, Nx = view.shape[:2]
    top_lefts = [(y0 + iy, x0 + ix) for iy in range(Ny) for ix in range(Nx)]
    return view, top_lefts

def topk_similar_torch(view: np.ndarray,
                       anchor_patch: np.ndarray,
                       K: int,
                       mean_normalize: bool = True,
                       device: str = DEVICE) -> np.ndarray:
    Ny, Nx, a, b = view.shape
    P = a * a
    V = view.reshape(Ny * Nx, P).astype(np.float32, copy=False)
    A = anchor_patch.reshape(P).astype(np.float32, copy=False)

    Vg = torch.from_numpy(V).to(device)
    Ag = torch.from_numpy(A).to(device)

    if mean_normalize:
        Vg = Vg - Vg.mean(dim=1, keepdim=True)
        Ag = Ag - Ag.mean()

    # squared distances
    d2 = ((Vg - Ag[None, :])**2).sum(dim=1)
    K_eff = int(min(K, d2.numel()))
    vals, idx = torch.topk(d2, k=K_eff, largest=False, sorted=True)
    return idx.detach().cpu().numpy()  # sorted ascending

def center_for_svd(M: np.ndarray, mode: str):
    """Center group matrix M (P×K); return (M_centered, ('mode', payload))."""
    if mode == "group":
        mu = M.mean(axis=1, keepdims=True)  # P×1
        return M - mu, ("group", mu)
    elif mode == "patch":
        mu = M.mean(axis=0, keepdims=True)  # 1×K
        return M - mu, ("patch", mu)
    return M, ("none", None)


## GPU SVD + σ-aware shrink

`svd_shrink_torch(M_cpu, method, tau, wnnm_c, sigma, K_eff)`

- Runs economy **SVD** on CUDA: `M = U S Vᵀ`.
- Shrink per component **on GPU**, then reconstruct:
  - **Soft (NNM)**: `S' = max(S − τ, 0)`
  - **WNNM**: `S' = max(S − (c·2σ²/(S+ε)), 0)`
  - **Wiener (PCA-Wiener)**: per-component **gain**  
    \( g_i = \frac{S_i^2}{S_i^2 + K\,\sigma^2} \) and `S' = g ⊙ S`
- Returns `M̂` on CPU (`float32`). If CUDA fails, falls back to CPU SVD for that group.


In [7]:
def svd_shrink_torch(M_cpu: np.ndarray,
                     method: str,
                     tau: float,
                     wnnm_c: float,
                     sigma: float,
                     K_eff: int,
                     device: str = DEVICE) -> np.ndarray:
    method = method.lower()

    def _soft(S_t):
        return torch.clamp(S_t - tau, min=0.0)

    def _wnnm(S_t):
        T = (wnnm_c * 2.0 * (sigma**2)) / (S_t + 1e-8)
        return torch.clamp(S_t - T, min=0.0)

    def _wiener(S_t):
        # PCA-Wiener gain per component using observed singular energy
        gain = (S_t*S_t) / (S_t*S_t + (K_eff * (sigma**2)))
        return gain * S_t

    # Try GPU first
    try:
        M = torch.from_numpy(M_cpu).to(device, dtype=torch.float32, non_blocking=True)
        U, S, Vh = torch.linalg.svd(M, full_matrices=False)

        if method in ("soft", "nnm"):
            S_shr = _soft(S)
        elif method == "wnnm":
            S_shr = _wnnm(S)
        elif method == "wiener":
            S_shr = _wiener(S)
        else:
            raise ValueError(f"Unknown method: {method}")

        Mhat = (U * S_shr) @ Vh
        return Mhat.detach().cpu().numpy().astype(np.float32, copy=False)
    except Exception:
        # CPU fallback (rare)
        U, S, VT = np.linalg.svd(M_cpu, full_matrices=False)
        if method in ("soft", "nnm"):
            S_shr = np.maximum(S - tau, 0.0, dtype=np.float32)
        elif method == "wnnm":
            T = (wnnm_c * 2.0 * (sigma**2)) / (S + 1e-8)
            S_shr = np.maximum(S - T.astype(np.float32), 0.0, dtype=np.float32)
        elif method == "wiener":
            gain = (S*S) / (S*S + (K_eff * (sigma**2)))
            S_shr = gain * S
        else:
            raise
        return ((U * S_shr) @ VT).astype(np.float32, copy=False)


## Main: `denoise_image_gpu(img, cfg)`

- Ensures input is 2D `float32` in `[0,1]` (RGB → luminance).
- Auto-estimates `sigma` & `tau` if not set.
- For each anchor: k-NN (GPU), SVD (GPU), shrink, reconstruct, **Kaiser** overlap-add.
- Optional: drop worst neighbors via `trim_frac` to resist outliers.

**Outputs**: `(denoised, meta)` where `meta = {'sigma','tau','method','device'}`


In [8]:
def denoise_image_gpu(img_in: np.ndarray, cfg: dict) -> tuple[np.ndarray, dict]:
    a = int(cfg["patch"])
    r = int(cfg["search_radius"])
    K = int(cfg["K"])
    step = int(cfg["anchor_step"])
    method = str(cfg["method"]).lower()
    tau = cfg["tau"]
    wnnm_c = float(cfg["wnnm_c"])
    sigma = cfg["sigma"]
    mean_norm = bool(cfg["mean_normalize"])
    center_mode = str(cfg["center_mode"]).lower()
    beta = float(cfg["kaiser_beta"])
    trim_frac = float(cfg.get("trim_frac", 0.0))
    dtype = cfg["dtype"]

    # Prepare image
    img = img_in
    if img.ndim == 3:
        img = _to_gray(img)
    if img_as_float32 is not None and img.dtype != np.float32:
        try:
            img = img_as_float32(img)
        except Exception:
            img = img.astype(np.float32, copy=False)
    else:
        img = img.astype(np.float32, copy=False)
    img = np.clip(img, 0.0, 1.0)

    H, W = img.shape
    if sigma is None:
        sigma = estimate_sigma(img)

    P = a * a
    if tau is None:
        tau = float(sigma * math.sqrt(2.0 * max(P, K)))

    wpatch = kaiser2d(a, beta=beta, dtype=dtype)
    num = np.zeros((H, W), dtype=dtype)
    den = np.zeros((H, W), dtype=dtype)

    ys = range(0, H - a + 1, step)
    xs = range(0, W - a + 1, step)

    for y in tqdm(ys, total=len(range(0, H - a + 1, step)), desc="Rows"):
        for x in xs:
            anchor = img[y:y+a, x:x+a].astype(np.float32, copy=False)

            # Candidates in local window
            view, top_lefts = extract_window_patches(img, y, x, a, r)
            Ny, Nx = view.shape[:2]
            V = view.reshape(Ny * Nx, a*a)

            # k-NN on GPU
            idx = topk_similar_torch(view, anchor, K, mean_normalize=mean_norm, device=DEVICE)
            if idx.size == 0:
                continue

            # Optional: trim worst neighbors (e.g., keep 90%)
            if trim_frac > 0.0:
                keep = max(1, int(round((1.0 - trim_frac) * idx.size)))
                idx = idx[:keep]

            K_eff = int(idx.size)

            # Build group
            group = V[idx].T  # (P, K_eff)

            # Centering
            group_c, payload = center_for_svd(group, center_mode)

            # GPU SVD shrink
            Mhat = svd_shrink_torch(group_c.astype(np.float32, copy=False),
                                    method, float(tau), float(wnnm_c), float(sigma),
                                    K_eff=K_eff, device=DEVICE)

            # Undo centering
            mode, mu = payload
            if mode == "group":
                Mhat = Mhat + mu
            elif mode == "patch":
                Mhat = Mhat + mu

            # Aggregate
            for k_local, flat_idx in enumerate(idx):
                yy, xx = top_lefts[flat_idx]
                patch_hat = Mhat[:, k_local].reshape(a, a)
                num[yy:yy+a, xx:xx+a] += patch_hat * wpatch
                den[yy:yy+a, xx:xx+a] += wpatch

    out = np.zeros_like(img, dtype=dtype)
    mask = den > 0
    out[mask] = num[mask] / den[mask]
    out[~mask] = img[~mask]
    out = np.clip(out, 0.0, 1.0)

    meta = {"sigma": float(sigma), "tau": float(tau), "method": method, "device": DEVICE}
    return out, meta


## Interactive UI (GPU) **with Save**

**Controls**
- `K`, `search_radius`, `anchor_step`
- `auto τ` or manual `τ`
- `method`: `'soft'`, `'wnnm'`, `'wiener'`
- `wnnm_c`: only for WNNM
- `center_mode`: `'patch'`, `'group'`, `'none'`
- `trim_frac`: drop worst fraction of neighbors after sort (0–0.3)
- **Output path** + **preserve float32** (TIFF via `tifffile` if available)

**Requirements**
- Define **`img0`** beforehand (2D float32 in `[0,1]`).  
- The cell runs **only** on your image — no demos/fallbacks.


In [9]:
import ipywidgets as W
from datetime import datetime

if "img0" not in globals() or img0 is None:
    raise RuntimeError("Please define `img0` (2D float32 in [0,1]) before running the UI.")

# State
_last = {"out": None, "meta": None, "params": None}

# Widgets
K_w           = W.IntSlider(cfg["K"], 8, 64, 1, description="K")
r_w           = W.IntSlider(cfg["search_radius"], 8, 48, 1, description="search_r")
step_w        = W.IntSlider(cfg["anchor_step"], 2, 12, 1, description="anchor_step")

tau_auto_w    = W.Checkbox(True, description="auto τ")
tau_w         = W.FloatLogSlider(value=0.02, base=10, min=-3, max=0, step=0.05, description="τ")

method_w      = W.Dropdown(options=["soft", "wnnm", "wiener"], value=cfg["method"], description="method")
wnnm_c_w      = W.FloatSlider(value=cfg["wnnm_c"], min=0.4, max=1.8, step=0.05, readout_format=".2f", description="wnnm_c")

center_w      = W.Dropdown(options=["patch","group","none"], value=cfg["center_mode"], description="center")
trim_w        = W.FloatSlider(value=cfg["trim_frac"], min=0.0, max=0.3, step=0.02, readout_format=".2f", description="trim_frac")

default_name  = os.path.join(DATA_DIR, f"denoised_gpu_{datetime.now().strftime('%Y%m%d_%H%M%S')}.tif")
out_path_w    = W.Text(value=default_name, description="Output", layout=W.Layout(width="70%"))
preserve_w    = W.Checkbox(value=True, description="Preserve float32 (TIFF)")

run_btn       = W.Button(description="Denoise (GPU)", button_style="primary")
save_btn      = W.Button(description="Save", button_style="success")
status_out    = W.Output()

def _run(_=None):
    with status_out:
        status_out.clear_output()
        local = cfg.copy()
        local.update(dict(
            K=int(K_w.value),
            search_radius=int(r_w.value),
            anchor_step=int(step_w.value),
            method=str(method_w.value),
            wnnm_c=float(wnnm_c_w.value),
            tau=None if tau_auto_w.value else float(tau_w.value),
            center_mode=str(center_w.value),
            trim_frac=float(trim_w.value),
        ))
        t0 = time.time()
        out, meta = denoise_image_gpu(img0, local)
        dt = time.time() - t0
        _last["out"], _last["meta"], _last["params"] = out, meta, local

        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1); plt.imshow(img0, cmap="gray"); plt.title("Input"); plt.axis("off")
        plt.subplot(1,2,2); plt.imshow(out,  cmap="gray"); plt.title(f"Denoised ({local['method']})"); plt.axis("off")
        plt.show()
        print(f"σ̂ {meta['sigma']:.4f} | τ {meta['tau']:.4f} | method {meta['method']} | time {dt:.2f}s")

def _save(_=None):
    with status_out:
        if _last["out"] is None:
            print("No result to save — click Denoise first.")
            return
        path = out_path_w.value.strip()
        if not path:
            print("Provide an output path.")
            return
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        img = _last["out"]
        ext = os.path.splitext(path)[1].lower()
        try:
            if ext in (".tif",".tiff"):
                if preserve_w.value:
                    try:
                        import tifffile as tiff
                        tiff.imwrite(path, img.astype(np.float32, copy=False))
                    except Exception as e:
                        from skimage import util as skut
                        print("[warn] tifffile unavailable; saving 16-bit:", e)
                        io.imsave(path, skut.img_as_uint(img))
                else:
                    from skimage import util as skut
                    io.imsave(path, skut.img_as_uint(img))
            else:
                from skimage import util as skut
                io.imsave(path, skut.img_as_ubyte(img))
            print("Saved:", path)
        except Exception as e:
            print("Save failed:", e)

def _toggle_tau(change):
    tau_w.disabled = change["new"]

run_btn.on_click(_run)
save_btn.on_click(_save)
tau_auto_w.observe(_toggle_tau, names="value")
tau_w.disabled = tau_auto_w.value

ui = W.VBox([
    W.HBox([K_w, r_w, step_w]),
    W.HBox([method_w, wnnm_c_w, center_w]),
    W.HBox([tau_auto_w, tau_w, trim_w]),
    W.HBox([out_path_w]),
    W.HBox([preserve_w, run_btn, save_btn]),
    status_out
])
display(ui)

print("Interactive GPU UI ready. Ensure `img0` is set (2D float32 in [0,1]).")


Interactive GPU UI ready. Ensure `img0` is set (2D float32 in [0,1]).
